# ДЗ 6. Очистка orders.csv в Pandas
Срок: **до воскресенья 22.11, 24:00**.
Сдача: папка `hw06` репозитория `dsa-homework` (`hw06.ipynb`): `git add hw06` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число и вывод для менеджера** (в print или Markdown-ячейке). Менеджер не читает код: вывод отвечает, что видно в данных и что делать.

Инструменты: всё с уроков 1–6. `groupby`, `pivot_table`, `merge` пока не используем (урок 7). Небольшой цикл по городам или месяцам допустим. Если берёте непройденный инструмент, объясните его в комментарии.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import pandas as pd
# 1
s = pd.Series([1, None, 3])
print(s.isna().sum(), s.fillna(0).mean())
# 2
print(pd.Series([" a", "A ", "b"]).str.strip().str.lower().nunique())
# 3
df = pd.DataFrame({"k": [1, 1, 2]})
df.drop_duplicates()
print(len(df))
# 4
print(pd.to_numeric(pd.Series(["1 000", "5"]).str.replace(" ", "")).sum())
# 5
d = pd.to_datetime(pd.Series(["2026-09-30 21:00", "2026-10-01 08:00"]))
print(d.dt.month.tolist())
```

Ваши ответы и объяснения:

1. `1 1.333333333333` - Метод `isna()` превращает `None` (в pandas становится `NaN`) в `True`, поэтому `.sum()` находит ровно 1 пропуск. Далее `fillna(0)` заменяет этот пропуск нулем, превращая серию в `[1.0, 0.0, 3.0]`, а `.mean()` вычисляет среднее арифметическое: (1 + 0 + 3) / 3 = 1.33333... .
2. `2` - метод `str.strip()` удаляет пробелы по краям, `str.lower()` переводить символов в нижний регистр: `["a", "a", "b"]`. А метод `nunique()` насчитывает уникальных значения (`"a"` и `"b"`). Их только 2.
3. `3` - Метод `drop_dublicates()` не изменяет исходный DataFrame если не задать его параметр `inplace=True`, поэтому таблица сохранила свои первоначальные 3 строки.
4. `1005` - Сначала `str.replace(" ", "")` удаляет пробел внутри строки: `"1 000"` -> `"1000"`. Затем `pd.to_numeric()` преобразует значении списка в целочисленный тип (int64), после чего `.sum()` складывает их как числа, выдавая 1005.
5. `[9, 10]` - `pd.to_datetime()` приводить строки в формат даты и времени - `datetime64`. `dt.month` извлекает номер месяца (9 и 10), а `tolist()` преобразует их в python список.

In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

orders.csv на месте: True


## [База] 1. Функция очистки
Напишите функцию `clean(path)`, которая читает orders.csv и делает все шаги урока: суммы в числа, города через словарь, дубли, даты, без TEST. Отменённые заказы **не** удаляйте: они нужны для доли отмен.
После каждого шага функция печатает, сколько строк осталось. Проверки внутри функции: у `amount` нет NaN, у `city` нет NaN, `order_id` уникален.
Сколько строк в результате?

In [3]:
# 1
import pandas as pd

def clean(path):
    # Читаем данные
    df = pd.read_csv(path)
    print("Количество строк:", len(df))
    
    # Суммы в числа
    df["amount"] = pd.to_numeric(df["amount"].str.replace(" ", ""))
    print("Количество строк после очистки amount:", len(df))
    
    # Города через словарь
    CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}
    df["city"] = df["city"].str.strip().str.lower().map(CITY)
    print("Количество строк после нормализации city:", len(df))
    
    # Удаляем дубли
    df = df.drop_duplicates()
    print("Количество строк после удаления дублей:", len(df))
    
    # Преобразуем даты
    df["order_date"] = pd.to_datetime(df["order_date"])
    print("Количество строк после преобразования дат:", len(df))
    
    # Убираем TEST
    df = df[df["customer_id"] != "TEST"]
    print("Количество строк после удаления TEST:", len(df))
    
    # Проверки
    if (df["amount"].isna().sum() == 0) & (df["city"].isna().sum() == 0) & (df["order_id"].is_unique):
        print("\nПроверки пройдены!")
    
    print("Итоговое количество строк:", len(df))
    
    return df


df_clean = clean("orders.csv")

Количество строк: 5463
Количество строк после очистки amount: 5463
Количество строк после нормализации city: 5463
Количество строк после удаления дублей: 5283
Количество строк после преобразования дат: 5283
Количество строк после удаления TEST: 5280

Проверки пройдены!
Итоговое количество строк: 5280


**Вывод:** 
- из 5 463 исходных записей получено 5 280 чистых заказов. Удалено 180 дублирующихся записей и 3 тестовых заказа. Суммы преобразованы в числовой формат, написания городов унифицированы, даты преобразованы в формат datetime.
- Отменённые заказы сохранены, поскольку они необходимы для дальнейшего анализа доли отмен.
- Проверки подтвердили, что пропусков в amount и city нет, а order_id уникален.

## [База] 2. Выручка по чистым данным
Выручка оплаченных заказов за сентябрь и октябрь и изменение в %. Затем то же для каждого из 6 городов (цикл по городам).
Совпадает ли с таблицей «город × месяц» урока 4? Какой город падает сильнее всех?

In [5]:
# 2
paid = df_clean[df_clean["status"] == "оплачен"].copy()

paid["month"] = paid["order_date"].dt.month

sep = paid[paid["month"] == 9]["amount"].sum()
octo = paid[paid["month"] == 10]["amount"].sum()

change = (octo - sep) / sep * 100

print("ОБЩАЯ ВЫРУЧКА (оплаченных заказов)")
print(f"Сентябрь: {sep:,.0f} ₸")
print(f"Октябрь: {octo:,.0f} ₸")
print(f"Изменение: {change:.2f}%")

print("\nВЫРУЧКА ПО ГОРОДАМ")

# Список шести городов
cities = [
    "Актобе", "Алматы", "Астана",
    "Атырау", "Караганда", "Шымкент"
]

for city in cities:

    # Выручка города за сентябрь
    sep_city = paid[(paid["city"] == city) & (paidb["month"] == 9)]["amount"].sum()

    # Выручка города за октябрь
    oct_city = paid[(paid["city"] == city) & (paid["month"] == 10)]["amount"].sum()

    # Процент изменения
    change_city = (oct_city - sep_city) / sep_city * 100

    print(f"\n{city}")
    print(f"Сентябрь: {sep_city:,.0f} ₸")
    print(f"Октябрь: {oct_city:,.0f} ₸")
    print(f"Изменение: {change_city:+.2f}%")

ОБЩАЯ ВЫРУЧКА (оплаченных заказов)
Сентябрь: 35,243,410 ₸
Октябрь: 34,419,520 ₸
Изменение: -2.34%

ВЫРУЧКА ПО ГОРОДАМ

Актобе
Сентябрь: 2,576,960 ₸
Октябрь: 2,540,420 ₸
Изменение: -1.42%

Алматы
Сентябрь: 14,501,070 ₸
Октябрь: 14,643,020 ₸
Изменение: +0.98%

Астана
Сентябрь: 9,034,220 ₸
Октябрь: 8,850,530 ₸
Изменение: -2.03%

Атырау
Сентябрь: 923,990 ₸
Октябрь: 969,860 ₸
Изменение: +4.96%

Караганда
Сентябрь: 3,648,520 ₸
Октябрь: 3,371,680 ₸
Изменение: -7.59%

Шымкент
Сентябрь: 4,558,650 ₸
Октябрь: 4,044,010 ₸
Изменение: -11.29%


**Вывод:** общая выручка оплаченных заказов в октябре снизилась на 2,34% по сравнению с сентябрём (на 823 890 ₸).
- Самое сильное падение произошло в Шымкенте — 11,29%.
- Караганда также показала существенное снижение — 7,59%.
- Алматы вырос на 0,98%.
- Атырау показал наибольший рост — 4,96%.

Стоит обратить внимание на снижение выручки в Шымкенте и Караганде. Необходимо проверить причины падения продаж в этих городах и определить, связано ли оно с уменьшением количества заказов или среднего чека.

## [База] 3. Журнал очистки для менеджера
Markdown-таблица: шаг очистки → сколько строк убрал → как изменилось «падение выручки» (от −22,7% до −2,3%).
Одной фразой для менеджера: какой шаг сильнее всего исказил его цифру?

In [8]:
# 3

### Журнал очистки данных

| № | Шаг очистки | Удалено строк | Падение выручки |
|---:|---|---:|---:|
| 1 | Исходные данные | 0 | -22,7% |
| 2 | Преобразование `amount` в числа | 0 | -22,7% |
| 3 | Нормализация городов | 0 | -22,7% |
| 4 | Удаление дубликатов | 180 | -19,6% |
| 5 | Преобразование дат | 0 | -19,6% |
| 6 | Удаление тестовых заказов `TEST` | 3 | -2,3% |
| **Итого** | **После очистки** | **183** | **-2,3%** |

**Вывод:** сильнее всего на показатель повлияло удаление 3 тестовых заказов: падение выручки изменилось с −19,6% до −2,3%, то есть исходные данные существенно завышали впечатление о снижении продаж.

## [Стандарт] 4. Пропуски в сроках доставки
Сколько пропусков в `delivery_days` по чистым данным? В каких городах и статусах их больше?
Доля доставки за 2 дня в Шымкенте: (а) `dropna`, (б) `fillna(медиана)`, (в) `fillna(0)`. Какой вариант честный и почему другие вводят в заблуждение?

In [10]:
# 4
# Пропуски в сроках доставки
# 1. Общее количество пропусков
missing = df_clean["delivery_days"].isna().sum()

print("Пропусков:", missing)
print("Доля пропусков:",
      round(missing / len(df_clean) * 100, 2), "%")

# 2. Пропуски по городам
print("\nПропуски по городам:")
print(df_clean[df_clean["delivery_days"].isna()]
      ["city"].value_counts())

# 3. Пропуски по статусам
print("\nПропуски по статусам:")
print(df_clean[df_clean["delivery_days"].isna()]
      ["status"].value_counts())

# 4. Доля доставки за 2 дня в Шымкенте
shymkent = df_clean[df_clean["city"] == "Шымкент"]

# А. Удаляем пропуски
a = shymkent["delivery_days"].dropna()
result_a = (a == 2).mean() * 100

# Б. Заполняем медианой всего датасета
median = df_clean["delivery_days"].median()
b = shymkent["delivery_days"].fillna(median)
result_b = (b == 2).mean() * 100

# В. Заполняем нулём
c = shymkent["delivery_days"].fillna(0)
result_c = (c == 2).mean() * 100

print("\nШЫМКЕНТ: ДОСТАВКА ЗА 2 ДНЯ")
print(f"А. dropna: {result_a:.2f}%")
print(f"Б. fillna(median={median}): {result_b:.2f}%")
print(f"В. fillna(0): {result_c:.2f}%")

Пропусков: 158
Доля пропусков: 2.99 %

Пропуски по городам:
Алматы       63
Астана       40
Караганда    24
Шымкент      22
Атырау        5
Актобе        4
Name: city, dtype: int64

Пропуски по статусам:
оплачен    144
отменён     14
Name: status, dtype: int64

ШЫМКЕНТ: ДОСТАВКА ЗА 2 ДНЯ
А. dropna: 14.85%
Б. fillna(median=2.0): 17.27%
В. fillna(0): 14.43%


**Вывод:** в данных обнаружено 158 пропусков в сроках доставки (2,99%). Больше всего пропусков в Алматы (63) и Астане (40). По статусам преобладают оплаченные заказы (144).

В Шымкенте доля доставки за 2 дня составляет 14,85% при исключении пропусков.
- dropna — наиболее честный вариант для оценки фактической доли среди заказов с известным сроком.
- fillna(медиана) завышает показатель до 17,27%, поскольку искусственно присваивает 22 заказам срок в 2 дня.
- fillna(0) занижает показатель до 14,43%, поскольку ноль не является реальным сроком доставки.

## [Стандарт] 5. Исправьте код от AI
AI написал «очистку» и получил 10 городов вместо 6, а строк осталось 5 463. В коде **две** ошибки. Найдите их и исправьте.
```python
df = pd.read_csv("orders.csv")
df.drop_duplicates()
df["city"] = df["city"].str.lower()
print(df["city"].nunique(), len(df))
```

In [14]:
# 5
df = pd.read_csv("orders.csv")
df = df.drop_duplicates(subset="order_id")
df["city"] = df["city"].str.strip().str.lower()

city_map = {
    "алматы": "Алматы",
    "almaty": "Алматы",
    "астана": "Астана",
    "astana": "Астана",
    "шымкент": "Шымкент",
    "shymkent": "Шымкент",
    "караганда": "Караганда",
    "актобе": "Актобе",
    "атырау": "Атырау"
}

df["city"] = df["city"].replace(city_map)

print("Количество городов:", df["city"].nunique())
print("Количество строк:", len(df))

Количество городов: 6
Количество строк: 5283


**Вывод:** AI допустил две ошибки: не сохранил результат удаления дубликатов и использовал только перевод названий в нижний регистр вместо полноценной нормализации. После исправления удалено 180 дублей, а 14 вариантов написания городов объединены в 6 корректных названий.

## [Стандарт] 6. Дни недели
По чистым данным: сколько заказов в каждый день недели (`.dt.day_name()` и `value_counts`)? В какой день заказов меньше всего и на сколько процентов меньше, чем в самый загруженный?
Что это значит для графика работы курьеров?

In [16]:
# 6
# Получаем названия дней недели
days = df_clean["order_date"].dt.day_name()

# Считаем количество заказов по дням
day_counts = days.value_counts()

print("Количество заказов по дням недели:")
print(day_counts)

# Самый загруженный и самый спокойный дни
max_day = day_counts.idxmax()
min_day = day_counts.idxmin()

max_orders = day_counts.max()
min_orders = day_counts.min()

# Процент разницы
difference = (max_orders - min_orders) / max_orders * 100

print("\nСамый загруженный день:", max_day, "-", max_orders)
print("Самый спокойный день:", min_day, "-", min_orders)
print(f"Разница: {difference:.2f}%")

Количество заказов по дням недели:
Thursday     786
Tuesday      785
Friday       781
Wednesday    774
Saturday     773
Sunday       697
Monday       684
Name: order_date, dtype: int64

Самый загруженный день: Thursday - 786
Самый спокойный день: Monday - 684
Разница: 12.98%


**Вывод:** наибольшая активность заказов наблюдается в четверг (786 заказов), наименьшая — в понедельник (684 заказа). Разница составляет 12,98%.
- график работы курьеров следует планировать с учётом фактической нагрузки по дням недели, усиливая смены в четверг (больше курьеров) и оптимизируя нагрузку в понедельник (более компактный график смен).
- Во вторник и пятницу также поддерживать повышенную готовность курьеров.
- Не сокращать количество курьеров слишком сильно в выходные: спрос остаётся достаточно высоким.

## [Челлендж] 7. Когда сломалась доставка в Шымкенте
Средний срок доставки в Шымкенте в сентябре и в октябре. Найдите первую дату, с которой срок доставки в Шымкенте не бывает меньше 5 дней (фильтр + `sort_values` по дате).
Проверьте, что в других городах в эту дату ничего не изменилось. Напишите менеджеру: что случилось, когда и что проверить в первую очередь.

In [25]:
# 7
# Оставляем заказы Шымкента
shymkent = df_clean[df_clean["city"] == "Шымкент"].copy()

# Средний срок доставки за сентябрь
sep = shymkent[shymkent["order_date"].dt.month == 9]["delivery_days"].mean()

# Средний срок доставки за октябрь
oct = shymkent[shymkent["order_date"].dt.month == 10]["delivery_days"].mean()

print("Средняя доставка в Шымкенте:")
print(f"Сентябрь: {sep:.2f} дня")
print(f"Октябрь: {oct:.2f} дня")

# Сортируем заказы по дате
shymkent = shymkent.sort_values("order_date")

# Ищем первую дату, после которой
# все известные сроки доставки >= 5 дней
dates = shymkent["order_date"].dt.date.unique()

for date in dates:
    later = shymkent[shymkent["order_date"].dt.date >= date]["delivery_days"].dropna()

    if len(later) > 0 and (later >= 5).all():
        print("\nДата изменения:", date)
        break

# Проверяем данные в остальных городах
print("\nПроверка других городов:")

for city in df_clean["city"].unique():
    if city != "Шымкент":

        city_data = df_clean[df_clean["city"] == city]

        sep_city = city_data[city_data["order_date"].dt.month == 9]["delivery_days"].mean()

        oct_city = city_data[city_data["order_date"].dt.month == 10]["delivery_days"].mean()

        print(f"{city}: сентябрь {sep_city:.2f}, ", f"октябрь {oct_city:.2f}")

Средняя доставка в Шымкенте:
Сентябрь: 2.03 дня
Октябрь: 6.54 дня

Дата изменения: 2026-10-01

Проверка других городов:
Алматы: сентябрь 1.98,  октябрь 2.01
Астана: сентябрь 2.05,  октябрь 1.99
Караганда: сентябрь 1.90,  октябрь 1.99
Атырау: сентябрь 2.06,  октябрь 2.01
Актобе: сентябрь 2.04,  октябрь 2.10


**Вывод:**

С 1 октября 2026 года в Шымкенте произошло резкое ухудшение сроков доставки. Среднее время доставки увеличилось с 2,03 до 6,54 дня, то есть более чем втрое.

Проверка остальных пяти городов показала, что аналогичного ухудшения нет. Следовательно, проблема, вероятнее всего, связана с локальными процессами доставки в Шымкенте.

Что необходимо проверить в первую очередь:
1. Работу местного склада и скорость обработки заказов.
2. Количество курьеров и изменения графика их работы с 1 октября.
3. Изменения маршрутов доставки и логистического партнёра.
4. Возможные технические сбои или изменения в системе учёта сроков доставки.

## [Челлендж] 8. Надёжный перевод сумм
Напишите функцию `to_amount(s)` для Series строк, которая понимает `"15 990"`, `"15990.0"`, `"1 200,50"` (запятая как разделитель дробной части) и превращает всё остальное (`"нет данных"`) в NaN.
Проверьте на `pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])` и на столбце `amount` из orders.csv: сумма должна совпасть с уроком.

In [30]:
# 8
def to_amount(s):
    s = (s.astype("string").str.replace(r"\s+", "", regex=True).str.replace(",", ".", regex=False))

    return pd.to_numeric(s, errors="coerce")

test = pd.Series(["15 990", "15990.0", "1 200,50", "нет данных"])
result = to_amount(test)
print(result)

# Применяем функцию к столбцу amount
df["amount"] = to_amount(df["amount"])

print("Сумма всех заказов:", df["amount"].sum())
print("Количество пропусков:", df["amount"].isna().sum())

# Дополнительно считаем оплаченные заказы
paid_sum = df.loc[df["status"] == "оплачен", "amount"].sum()

print("Сумма оплаченных заказов:", paid_sum)

0    15990.0
1    15990.0
2     1200.5
3        NaN
dtype: float64
Сумма всех заказов: 107441780.0
Количество пропусков: 0
Сумма оплаченных заказов: 99662900.0
